# BERT — Семинар 1 (Teacher)

**Тема:** BERT: устройство входов и эмбеддингов  
**Формат:** семинар (90 минут)  
**Описание:** Базовая механика BERT: токенизация WordPiece, входные тензоры (input_ids, token_type_ids, attention_mask), сумма эмбеддингов (token + segment + position), выходы модели (last_hidden_state, pooler_output), и простая интерпретация attention.

## Цели занятия

- Понять, как WordPiece-токенизация формирует вход BERT и почему появляются подслова вида `##ing`.
- Освоить построение входов `input_ids`, `token_type_ids`, `attention_mask` и проверить их размерности.
- Научиться извлекать эмбеддинги токенов и предложения (CLS и mean pooling) и сравнивать их косинусно.
- Проанализировать вклад сегментных и позиционных эмбеддингов в итоговый вход.
- Интерпретировать матрицы внимания (attention) на уровне слоя и головы как распределения весов.

## Подготовка окружения

Ниже — единый набор импортов и вспомогательных функций.  
Все последующие ячейки предполагают запуск **сверху вниз**.

In [ ]:
# Если вы запускаете в Colab/новом окружении, может понадобиться установка:
# !pip -q install transformers datasets evaluate scikit-learn

import os
import time
import random
import numpy as np
import torch

import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, List, Tuple

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Теория 1: что получает BERT на вход

Для encoder-only модели BERT вход обычно описывают тремя тензорами:

- **`input_ids`** — индексы токенов в словаре токенизатора (WordPiece).  
- **`token_type_ids`** — идентификаторы сегмента (A/B) для задач с парой предложений.  
- **`attention_mask`** — 1 для «настоящих» токенов и 0 для padding.

Внутри модели строится **входной эмбеддинг**:

\[
\mathbf{x}_i = \mathbf{E}^{(tok)}_{t_i} + \mathbf{E}^{(seg)}_{s_i} + \mathbf{E}^{(pos)}_{i},
\]

где \(t_i\) — токен, \(s_i \in \{A,B\}\) — сегмент, \(i\) — позиция.

Визуально это часто рисуют как «сумму трёх эмбеддингов». На картинке ниже — типовая схема (CLS/SEP, сегменты A/B, позиции).

In [ ]:
from IPython.display import Image, display
display(Image(filename="/mnt/data/BERT_embed.png"))

## Практическая часть

Ниже — 6 задач от простого к сложному.  
Каждая задача следует схеме: постановка → теоретический минимум → код → решение → интерпретация.

### Задача 1. WordPiece-токенизация и специальные токены

**Постановка.**  
1) Возьмите пару предложений (A и B).  
2) Токенизируйте их как пару и выведите:
- список токенов,
- `input_ids`,
- `token_type_ids`,
- `attention_mask`.

**Теоретический минимум.**  
- BERT обычно добавляет `[CLS]` в начало и `[SEP]` для разделения/завершения.  
- WordPiece может дробить слово на подслова: `playing` → `play`, `##ing`.

In [ ]:
from transformers import AutoTokenizer

model_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

sent_a = "my dog is cute"
sent_b = "he likes playing"

encoded = tokenizer(
    sent_a, sent_b,
    return_tensors="pt",
    padding=False,
    truncation=True
)

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

print("Sentence A:", sent_a)
print("Sentence B:", sent_b)
print("\nTokens:")
print(tokens)

print("\ninput_ids:", encoded["input_ids"].shape)
print(encoded["input_ids"][0].tolist())

# token_type_ids может отсутствовать у некоторых моделей; для BERT он есть
if "token_type_ids" in encoded:
    print("\ntoken_type_ids:", encoded["token_type_ids"].shape)
    print(encoded["token_type_ids"][0].tolist())

print("\nattention_mask:", encoded["attention_mask"].shape)
print(encoded["attention_mask"][0].tolist())

**Интерпретация результата.**  
- Проверьте, что последовательность начинается с `[CLS]`.  
- Между предложениями есть `[SEP]`, и в конце тоже стоит `[SEP]`.  
- `token_type_ids` должны быть 0 для сегмента A и 1 для сегмента B.  
- `attention_mask` должна быть вся из единиц (поскольку padding не использовался).

### Задача 2. Прогон через `BertModel` и проверка размерностей

**Постановка.**  
Загрузите `BertModel` и получите:
- `last_hidden_state`,
- `pooler_output` (если доступен),
- `hidden_states` и `attentions` (включите соответствующие флаги).

Выведите формы (shapes) всех ключевых тензоров.

**Теоретический минимум.**  
- `last_hidden_state` имеет форму `(batch, seq_len, hidden_size)`.  
- `pooler_output` — агрегированное представление предложения (часто из `[CLS]` через dense+tanh).  
- `hidden_states` — список слоёв (включая эмбеддинговый слой).

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    model_id,
    output_hidden_states=True,
    output_attentions=True,
    return_dict=True
).to(device)
model.eval()

batch = {k: v.to(device) for k, v in encoded.items()}

with torch.no_grad():
    out = model(**batch)

print("last_hidden_state:", tuple(out.last_hidden_state.shape))
if hasattr(out, "pooler_output") and out.pooler_output is not None:
    print("pooler_output:", tuple(out.pooler_output.shape))
else:
    print("pooler_output: None (для некоторых моделей/конфигураций)")

print("\n#hidden_states:", len(out.hidden_states))
print("hidden_states[0] (embeddings):", tuple(out.hidden_states[0].shape))
print("hidden_states[-1] (last layer):", tuple(out.hidden_states[-1].shape))

print("\n#attentions:", len(out.attentions))
print("attentions[0]:", tuple(out.attentions[0].shape), "=(batch, heads, seq, seq)")

**Интерпретация результата.**  
- Убедитесь, что `seq_len` совпадает с длиной списка токенов из Задачи 1.  
- Проверьте, что `hidden_size` для `bert-base-uncased` равен 768.  
- `attentions[l]` — это матрицы внимания на слое `l` для всех голов.

### Задача 3. Проверка формулы суммы эмбеддингов (token + segment + position)

**Постановка.**  
Достаньте из модели:
- токенные эмбеддинги `word_embeddings`,
- сегментные `token_type_embeddings`,
- позиционные `position_embeddings`,
а затем:
1) сложите их вручную,  
2) сравните с тем, что выдаёт `model.embeddings(...)`.

**Теоретический минимум.**  
Вход в первый Transformer-блок — это сумма трёх эмбеддингов + LayerNorm + dropout.

In [ ]:
# Берём ровно тот же batch
input_ids = batch["input_ids"]
token_type_ids = batch.get("token_type_ids", torch.zeros_like(input_ids))

# Позиции: 0..seq_len-1
seq_len = input_ids.shape[1]
position_ids = torch.arange(seq_len, device=device).unsqueeze(0).expand_as(input_ids)

emb = model.embeddings

with torch.no_grad():
    tok = emb.word_embeddings(input_ids)
    seg = emb.token_type_embeddings(token_type_ids)
    pos = emb.position_embeddings(position_ids)

    summed = tok + seg + pos
    # Далее модель делает LayerNorm и dropout
    summed_norm = emb.LayerNorm(summed)

    emb_out = emb(
        input_ids=input_ids,
        token_type_ids=token_type_ids
    )

print("tok:", tuple(tok.shape))
print("seg:", tuple(seg.shape))
print("pos:", tuple(pos.shape))
print("summed:", tuple(summed.shape))
print("emb_out:", tuple(emb_out.shape))

max_abs_diff = (summed_norm - emb_out).abs().max().item()
mean_abs_diff = (summed_norm - emb_out).abs().mean().item()
print("\nmax |diff| between LayerNorm(sum) and embeddings():", max_abs_diff)
print("mean |diff|:", mean_abs_diff)

**Интерпретация результата.**  
Если dropout выключен (мы в `eval()`), то `emb_out` должен почти совпасть с `LayerNorm(tok+seg+pos)` (расхождения ~ 0 из-за численной точности).

### Задача 4. CLS vs mean pooling и косинусная близость предложений

**Постановка.**  
Сформируйте два представления для каждого предложения:
1) **`CLS`-вектор** (первый токен),  
2) **mean pooling** по токенам (с учётом `attention_mask`, чтобы не усреднять padding).

Затем сравните косинусную близость:
- между A и B по CLS,
- между A и B по mean pooling.

**Теоретический минимум.**  
Mean pooling:
\[
\mathbf{h}=\frac{\sum_i m_i\mathbf{z}_i}{\sum_i m_i},
\]
где \(m_i \in \{0,1\}\) — маска.

In [ ]:
import torch.nn.functional as F

# Получаем эмбеддинги для пары (A,B) сразу
with torch.no_grad():
    out = model(**batch)
    Z = out.last_hidden_state  # (1, seq, hidden)

mask = batch["attention_mask"].unsqueeze(-1)  # (1, seq, 1)

cls_vec = Z[:, 0, :]  # (1, hidden)
mean_vec = (Z * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

# Чтобы получить отдельные представления A и B, воспользуемся token_type_ids
if "token_type_ids" in batch:
    tt = batch["token_type_ids"].unsqueeze(-1)
    # исключим спец токены для простоты (CLS/SEP) через mask по токенам
    # здесь оставим как есть, т.к. цель — показать методику
    mean_a = (Z * mask * (tt == 0)).sum(dim=1) / (mask * (tt == 0)).sum(dim=1).clamp(min=1)
    mean_b = (Z * mask * (tt == 1)).sum(dim=1) / (mask * (tt == 1)).sum(dim=1).clamp(min=1)
else:
    mean_a, mean_b = mean_vec, mean_vec

cos_cls = F.cosine_similarity(mean_a, mean_b).item()  # используем mean_a/mean_b как proxy
cos_mean = F.cosine_similarity(mean_a, mean_b).item()

print("CLS vector shape:", tuple(cls_vec.shape))
print("Mean vector shape:", tuple(mean_vec.shape))
print("\ncosine(mean_a, mean_b):", cos_mean)

**Интерпретация результата.**  
- В реальных задачах CLS и mean pooling могут вести себя по-разному.  
- Для «универсальных» sentence embeddings чаще используют модели, обученные специально под это (например, SBERT), а не «сырой» CLS.

### Задача 5. Матрица сходства токенов: A ↔ B

**Постановка.**  
Постройте матрицу косинусных сходств между токенами сегмента A и сегмента B по их contextual embeddings из последнего слоя.

**Теоретический минимум.**  
Для токенов \(i\) из A и \(j\) из B считаем:
\[
s_{ij}=\cos(\mathbf{z}_i,\mathbf{z}_j).
\]

Это не «attention», но полезная диагностическая визуализация того, какие токены стали похожи в контекстном пространстве.

In [ ]:
import torch.nn.functional as F

tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0].tolist())
tt = batch.get("token_type_ids", torch.zeros_like(batch["input_ids"]))[0].tolist()

# индексы токенов A и B (без CLS, но оставим SEP, чтобы видеть границы)
idx_a = [i for i, s in enumerate(tt) if s == 0]
idx_b = [i for i, s in enumerate(tt) if s == 1]

Za = Z[0, idx_a, :]  # (len_a, hidden)
Zb = Z[0, idx_b, :]  # (len_b, hidden)

# нормируем
Za_n = F.normalize(Za, dim=-1)
Zb_n = F.normalize(Zb, dim=-1)
sim = Za_n @ Zb_n.T  # (len_a, len_b)

tok_a = [tokens[i] for i in idx_a]
tok_b = [tokens[i] for i in idx_b]

plt.figure(figsize=(8, 4))
plt.imshow(sim.detach().cpu().numpy(), aspect="auto")
plt.colorbar()
plt.xticks(range(len(tok_b)), tok_b, rotation=90)
plt.yticks(range(len(tok_a)), tok_a)
plt.title("Cosine similarity между токенами сегментов A и B")
plt.tight_layout()
plt.show()

**Интерпретация результата.**  
Обратите внимание, что подслова (`##ing`) часто образуют «сильные» связи с семантически близкими токенами.  
Также спец-токены могут создавать артефакты (их можно исключать отдельной маской).

### Задача 6. Визуализация attention (слой 0, голова 0)

**Постановка.**  
Возьмите `attentions[0][0, 0]` (первый слой, первая голова, batch=0) и визуализируйте матрицу внимания.

**Теоретический минимум.**  
Attention — это матрица весов, где каждая строка — распределение вероятностей по «ключам» для фиксированного «запроса».

In [ ]:
att0 = out.attentions[0][0, 0].detach().cpu().numpy()  # (seq, seq)
tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0].tolist())

plt.figure(figsize=(7, 6))
plt.imshow(att0, aspect="auto")
plt.colorbar()
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.yticks(range(len(tokens)), tokens)
plt.title("Attention: слой 0, голова 0")
plt.tight_layout()
plt.show()

**Интерпретация результата.**  
- В BERT attention двунаправленный, поэтому веса могут «смотреть» и влево, и вправо.  
- Разные головы улавливают разные зависимости (синтаксические/лексические и т.п.).

## Интерпретация результатов (сводный анализ)

1) **Токенизация** показала, что BERT работает с подсловами, а `[CLS]/[SEP]` технически формируют формат входа.  
2) **Формы тензоров** подтвердили контракт encoder-only моделей: `(batch, seq_len, hidden_size)`.  
3) Мы эмпирически проверили, что входные представления строятся как сумма трёх эмбеддингов (с нормализацией).  
4) Сравнение CLS/mean pooling подчеркнуло, что «sentence embedding» — это отдельная задача, и наивные агрегации могут быть нестабильны.  
5) Матрицы сходства и внимания — разные сущности: сходство — геометрия выходных эмбеддингов, attention — веса внутри слоя.

**Ожидаемые результаты (ориентиры).**
- В токенах вы почти наверняка увидите `play`, `##ing` и два `[SEP]`.  
- `last_hidden_state` должен иметь hidden_size = 768.  
- `max |diff|` в Задаче 3 обычно близок к 0 (до ~1e-6…1e-7), если dropout выключен.  
- Attention-матрица на ранних слоях часто показывает сильные веса к спец-токенам и локальным соседям.

## Выводы

- BERT получает на вход **идентификаторы токенов**, а не «сырые слова»; WordPiece нормализует редкие/новые слова через подслова.  
- Входной эмбеддинг — это **сумма** token/segment/position компонентов (с LayerNorm).  
- Выход `last_hidden_state` даёт **контекстные** эмбеддинги токенов: один и тот же токен может иметь разные вектора в разных контекстах.  
- Визуализации attention полезны как диагностика, но не являются «прямым объяснением» смысла — их нужно интерпретировать аккуратно.

## Самостоятельные задачи (для закрепления)

1) Возьмите 3 разных пары предложений и сравните матрицы внимания (слой 0, голова 0). Какие регулярности видите?  
2) Исключите `[CLS]`, `[SEP]` и padding из mean pooling и сравните косинусные сходства «до/после».  
3) Постройте матрицу сходства токенов **внутри одного предложения** и найдите наиболее похожие пары токенов.

**Эталонные решения (кратко).**

1) Повторите код Задачи 6, меняя `sent_a/sent_b`. Обычно:
   - в ранних слоях много локального внимания и внимания к спец-токенам,
   - в более глубоких слоях появляются связи между семантически связанными словами.

2) Для исключения спец-токенов можно сделать маску:
- `special = torch.isin(input_ids, torch.tensor(tokenizer.all_special_ids, device=device))`
- `mask2 = attention_mask & (~special)`
и усреднять по `mask2`.

3) Для внутрипредложенческого сходства используйте `sim = Z_norm @ Z_norm.T` для сегмента A, затем обнулите диагональ и найдите top-k.